In [1]:
import json
import pandas as pd

In [32]:
gold_standard_dataset = pd.read_csv('gold_standard_dataset.csv')

contains_fields = gold_standard_dataset["field"].str.contains(
        r"time_seen_units|time_birth_units|systolic_blood_pressure|diastolic_blood_pressure", regex=True, na=False
    )
gold_standard_dataset = gold_standard_dataset[~contains_fields]
gold_standard_dataset.head()

,hospital,patient_id,form_type,page,field,value
0,76,76000172,NAR,2,appearance,well
1,76,76000172,NAR,2,capillary_refill_in_seconds,2
2,76,76000172,NAR,2,chest_indrawing,TRUE
3,76,76000172,NAR,2,given_chlorhexidine,TRUE
4,76,76000172,NAR,2,given_prophylaxis_pmtct,FALSE


In [31]:
contains_fields = gold_standard_dataset["field"].str.contains(
        r"systolic_blood_pressure|diastolic_blood_pressure", regex=True, na=False
    )
gold_standard_dataset[contains_fields]

,hospital,patient_id,form_type,page,field,value


In [15]:
# db.webui_form_processor_stats.findOne({ image_filename: "ITF_40000133_page_1.png" })

gold_standard_dataset[
(gold_standard_dataset["field"]=="pulse_oximetry") & (gold_standard_dataset["form_type"]=="ITF")]

,hospital,patient_id,form_type,page,field,value
2089,41,41000329,ITF,1,pulse_oximetry,99
2613,41,41000310,ITF,1,pulse_oximetry,96
3132,76,76000035,ITF,1,pulse_oximetry,72
3379,40,40000319,ITF,1,pulse_oximetry,95
3827,63,63000011,ITF,1,pulse_oximetry,96
...,...,...,...,...,...,...
285243,72,72000872,ITF,1,pulse_oximetry,45
285774,72,72000877,ITF,1,pulse_oximetry,95
286032,72,72000882,ITF,1,pulse_oximetry,98
286295,72,72000886,ITF,1,pulse_oximetry,96


In [23]:
gold_standard_dataset[
(gold_standard_dataset["field"]=="date") & (gold_standard_dataset["form_type"]=="NAR") & (gold_standard_dataset["page"]==1)]

,hospital,patient_id,form_type,page,field,value
44,76,76000172,NAR,1,date,24-03-2025
87,41,41000118,NAR,1,date,08-02-2025
254,52,52000025,NAR,1,date,06-01-2025
332,52,52000213,NAR,1,date,02-01-2025
397,52,52000518,NAR,1,date,25-03-2025
...,...,...,...,...,...,...
285823,72,72000878,NAR,1,date,12-01-2025
285913,72,72000880,NAR,1,date,11-01-2220
286047,72,72000882,NAR,1,date,12-01-2025
286179,72,72000883,NAR,1,date,18-01-2025


In [19]:
page_fields = gold_standard_dataset.groupby(["form_type", "page"])['field'].agg(lambda x: sorted(x.unique()))
gold_data_fields = {
    f"{form_type}_{page}": fields
    for (form_type, page), fields in page_fields.items()
}
gold_data_fields

{'ITF_1': ['abnormal_placenta',
  'anc_visits',
  'antenatal_steroids',
  'apgar_10m',
  'apgar_1m',
  'apgar_5m',
  'attended_anc',
  'baby_age',
  'baby_from',
  'birth_date',
  'birth_weight',
  'blood_group',
  'chest_compressions',
  'date_estimated_delivery_date',
  'date_last_menstrual_period',
  'delivery_type',
  'fetal_distress',
  'gestation_in_weeks',
  'given_bcg',
  'given_chlorhexidine',
  'given_teo',
  'given_vitamin_k',
  'gravida',
  'had_cs',
  'has_fever',
  'maternal_status',
  'multiple_pregnancy',
  'mum_age_in_years',
  'mum_had_antepartum_haemorrhage',
  'mum_had_diabetes',
  'mum_had_eclampsia',
  'mum_had_hep_b',
  'mum_had_hypertension_in_pregnancy',
  'mum_had_pre_eclampsia',
  'mum_had_vdrl',
  'mum_has_anc_ultrasound',
  'mum_on_arvs',
  'mum_pmtct_status',
  'mum_treated_for_tb',
  'parity_abortions',
  'parity_live',
  'passed_meconium',
  'placenta_complete',
  'prescribed_antibiotics',
  'prescribed_cpap',
  'prescribed_opv',
  'prescribed_oxygen',
 

In [10]:
with open("mongodb_key_structure.json", "r", encoding="utf-8") as f:
    llm_fields = json.load(f)

llm_fields

{'ITF_1': {'mother_details': ['antenatal_ultrasound_performed',
   'antepartum_hemorrhage',
   'current_maternal_medications',
   'date_form_completed',
   'eclampsia',
   'expected_date_of_delivery',
   'gestational_age_at_delivery_in_weeks',
   'hiv_status_prevention_of_mother-to-child_transmission',
   'hypertension_in_pregnancy',
   'last_menstrual_period',
   'maternal_fever_present',
   "mother's_age_in_years",
   "mother's_blood_group",
   'mother_attended_antenatal_care',
   'mother_has_diabetes',
   'mother_on_antibiotics',
   'mother_on_antiretroviral_therapy',
   'mother_on_tb_treatment',
   'multiple_pregnancy',
   'number_of_anc_visits_attended',
   'number_of_fetuses_in_multiple_pregnancy',
   'number_of_prior_pregnancies_eg_3+0',
   'pre-eclampsia',
   'rhesus_factor_status',
   'syphilis_screening_vdrl_test',
   'total_number_of_pregnancies'],
  'labour_birth': ['bag_and_mask_ventilation_given',
   'bcg_vaccine',
   'chest_compressions_performed',
   'chlorhexidine_cord

In [12]:
llm_gold_key_mapper = {}
llm_gold_key_mapper["ITF_1"]={
  'abnormal_placenta':'placental_abnormalities',
  'anc_visits':'number_of_anc_visits_attended',
  'antenatal_steroids':'corticosteroids_given',
  'apgar_10m':'apgar_score_at_10_minutes',
  'apgar_1m': 'apgar_score_at_1_minute',
  'apgar_5m':'apgar_score_at_5_minutes',
  'attended_anc':'mother_attended_antenatal_care',
  'baby_age':'neonatal_age',
  'baby_from':'location_baby_originated_from',
  'birth_date':"baby's_date_of_birth",
  'birth_weight':'birth_weight_in_grams',
  'blood_group':"mother's_blood_group",
  'chest_compressions':'chest_compressions_performed',
  'date_estimated_delivery_date':'expected_date_of_delivery',
  'date_last_menstrual_period': 'last_menstrual_period',
  'delivery_type':'mode_of_delivery',
  'fetal_distress':'fetal_distress_during_labour',
  'gestation_in_weeks':'gestational_age_at_delivery_in_weeks',
  'given_bcg':'bcg_vaccine',
  'given_chlorhexidine':'chlorhexidine_cord_care',
  'given_teo': 'thermal_care', # mis-labelled in original dataset
  'given_vitamin_k':'vitamin_k_prophylaxis_given',
  'gravida':'total_number_of_pregnancies',
  'had_cs':'type_of_caesarean_section',
  'has_fever':'maternal_fever_present',
  'maternal_status':"mother's_current_location",
  'multiple_pregnancy':'multiple_pregnancy',
  'mum_age_in_years': "mother's_age_in_years",
  'mum_had_antepartum_haemorrhage':'antepartum_hemorrhage',
  'mum_had_diabetes':'mother_has_diabetes',
  'mum_had_eclampsia':'eclampsia',
  'mum_had_hep_b':'hepatitis_b_vaccine',
  'mum_had_hypertension_in_pregnancy':'hypertension_in_pregnancy',
  'mum_had_pre_eclampsia':'pre-eclampsia',
  'mum_had_vdrl':'syphilis_screening_vdrl_test',
  'mum_has_anc_ultrasound':'antenatal_ultrasound_performed',
  'mum_on_arvs': 'mother_on_antiretroviral_therapy',
  'mum_pmtct_status':'hiv_status_prevention_of_mother-to-child_transmission',
  'mum_treated_for_tb':'mother_on_tb_treatment',
  'number_of_prior_pregnancies':'number_of_prior_pregnancies_eg_3+0',
  #'parity_abortions',
  #'parity_live',
  #'pulse_oximetry',
  'passed_meconium':'meconium-stained_amniotic_fluid',
  'placenta_complete':'placenta_completely_delivered',
  'prescribed_antibiotics':'mother_on_antibiotics',
  'prescribed_cpap':'cpap_support',
  'prescribed_opv':'oral_polio_vaccine',
  'prescribed_oxygen':'supplemental_oxygen',
  'pulse_rate':'heart_rate_beats_per_minute',
  'rapture_of_membrane':'rupture_of_membranes_timing_eg_>18h',
  'respiratory_rate':'respiratory_rate',
  'rhesus':'rhesus_factor_status',
  'sex':"baby's_sex",
  'temparature':'temperature_in_°c',
  'was_resuscitated':'bag_and_mask_ventilation_given',
  'weight':'current_weight_in_grams',
}

len(list(llm_gold_key_mapper['ITF_1'].keys()))

54

In [14]:
json.dumps(llm_gold_key_mapper['ITF_1'])

'{"abnormal_placenta": "placental_abnormalities", "anc_visits": "number_of_anc_visits_attended", "antenatal_steroids": "corticosteroids_given", "apgar_10m": "apgar_score_at_10_minutes", "apgar_1m": "apgar_score_at_1_minute", "apgar_5m": "apgar_score_at_5_minutes", "attended_anc": "mother_attended_antenatal_care", "baby_age": "neonatal_age", "baby_from": "location_baby_originated_from", "birth_date": "baby\'s_date_of_birth", "birth_weight": "birth_weight_in_grams", "blood_group": "mother\'s_blood_group", "chest_compressions": "chest_compressions_performed", "date_estimated_delivery_date": "expected_date_of_delivery", "date_last_menstrual_period": "last_menstrual_period", "delivery_type": "mode_of_delivery", "fetal_distress": "fetal_distress_during_labour", "gestation_in_weeks": "gestational_age_at_delivery_in_weeks", "given_bcg": "bcg_vaccine", "given_chlorhexidine": "chlorhexidine_cord_care", "given_teo": "thermal_care", "given_vitamin_k": "vitamin_k_prophylaxis_given", "gravida": "tot

In [33]:
llm_gold_key_mapper["NAR_1"]={
    'date':'date_infant_admitted',
    'anc_visits':'number_of_anc_visits_attended',
    'apgar_10m':'apgar_score_at_10_minutes',
    'apgar_1m':'apgar_score_at_1_minute',
    'apgar_5m':'apgar_score_at_5_minutes',
    'birth_date':"baby's_date_of_birth",
    'birth_weight':'birth_weight_in_grams',
    'blood_group':"mother's_blood_group",

    #'baby_age_in_days',
    #'born_before_arrival',
    #'mum_given_HBIG_treatment',
    #'mum_on_arvs',
    #'head_circumference',
    #'length',
    #'parity_abortions',
    #'parity_live',
    #'mum_had_hepatitis_b',

    'born_where':'if_baby_is_born_outside_facility',
    'date_estimated_delivery_date':'expected_date_of_delivery',
    'delivery_type':'mode_of_delivery',
    'gestation_in_weeks':'gestational_age_at_delivery_in_weeks',
    'gestation_type':'gestational_age_calculated_from',
    'given_anti_D_medication':'rhesus_anti-d_given',
    'had_cs':'type_of_caesarean_section',
    'has_apnoea':'baby_has_apnoea',
    'has_convulsions':'baby_has_convulsions',
    'has_diarhoea':'baby_has_bloody_stool',
    'has_difficulty_breathing':'baby_has_difficulty_breathing',
    'has_difficulty_feeding':'baby_has_difficulty_feeding',
    'has_fever':'baby_has_fever_present',
    'has_vomiting':'baby_has_bilious_vomiting',
    'is_floppy':'baby_is_floppy',
    'is_multiple_delivery':'multiple_deliveries',
    'multiple_delivery_num':'number_of_fetuses_in_multiple_pregnancy',
    'mum_age_in_years': "mother's_age_in_years",
    'mum_had_antepartum_haemorrhage':'antepartum_hemorrhage',
    'mum_had_diabetes':'mother_has_diabetes',
    'mum_had_hypertension_in_pregnancy':'hypertension_in_pregnancy',
    'mum_had_vdrl':'syphilis_screening_vdrl_test',
    'mum_has_anc_ultrasound':'antenatal_ultrasound_performed',
    'mum_pmtct_status':'hiv_status_prevention_of_mother-to-child_transmission',
    'passed_meconium':'baby_passed_meconium_stool',
    'passed_urine':'baby_passed_urine',
    'prolonged_labour':'mother_had_prolonged_labour',
    'pulse_oximetry':'oxygen_saturation',
    'pulse_rate':'heart_rate_beats_per_minute',
    'rapture_of_membrane':'rupture_of_membranes_timing_in_hours',
    'respiratory_rate':'respiratory_rate',
    'rhesus':'rhesus_factor_status',
    'sex':"baby's_sex",
    'temparature':'temperature_in_°c',
    'time_birth':"baby's_time_of_birth",
    'time_seen':'time_baby_seen',
    'was_resuscitated':'bag_and_mask_ventilation_given',
    'weight':'current_weight_in_grams'
}

json.dumps(llm_gold_key_mapper['NAR_1'])

'{"date": "date_infant_admitted", "anc_visits": "number_of_anc_visits_attended", "apgar_10m": "apgar_score_at_10_minutes", "apgar_1m": "apgar_score_at_1_minute", "apgar_5m": "apgar_score_at_5_minutes", "birth_date": "baby\'s_date_of_birth", "birth_weight": "birth_weight_in_grams", "blood_group": "mother\'s_blood_group", "born_where": "if_baby_is_born_outside_facility", "date_estimated_delivery_date": "expected_date_of_delivery", "delivery_type": "mode_of_delivery", "gestation_in_weeks": "gestational_age_at_delivery_in_weeks", "gestation_type": "gestational_age_calculated_from", "given_anti_D_medication": "rhesus_anti-d_given", "had_cs": "type_of_caesarean_section", "has_apnoea": "baby_has_apnoea", "has_convulsions": "baby_has_convulsions", "has_diarhoea": "baby_has_bloody_stool", "has_difficulty_breathing": "baby_has_difficulty_breathing", "has_difficulty_feeding": "baby_has_difficulty_feeding", "has_fever": "baby_has_fever_present", "has_vomiting": "baby_has_bilious_vomiting", "is_flo

In [17]:
llm_fields['NAR_1']

{'infant_details': ['apgar_score_at_10_minutes',
  'apgar_score_at_1_minute',
  'apgar_score_at_5_minutes',
  "baby's_date_of_birth",
  "baby's_sex",
  "baby's_time_of_birth",
  'bag_and_mask_ventilation_given',
  'date_infant_admitted',
  'gestational_age_at_delivery_in_weeks',
  'gestational_age_calculated_from',
  'if_baby_is_born_outside_facility',
  'mode_of_delivery',
  "mother's_age_in_years",
  'multiple_deliveries',
  'number_of_fetuses_in_multiple_pregnancy',
  'rupture_of_membranes_timing_in_hours',
  'time_baby_seen',
  'type_of_caesarean_section'],
 'mother_details': ['antenatal_ultrasound_performed',
  'antepartum_hemorrhage',
  'expected_date_of_delivery',
  'hiv_status_prevention_of_mother-to-child_transmission',
  'hypertension_in_pregnancy',
  "mother's_blood_group",
  'mother_had_prolonged_labour',
  'mother_has_diabetes',
  'number_of_anc_visits_attended',
  'number_of_prior_pregnancies_eg_3+0',
  'rhesus_anti-d_given',
  'rhesus_factor_status',
  'syphilis_screenin